# Normalización completa del dataset de incendios de Galicia

## 1. Importar librerías necesarias

In [1]:
import pandas as pd
import numpy as np
import unicodedata
import itertools
import datetime
import os

from pathlib import Path
from difflib import get_close_matches
from tqdm import tqdm

## 2. Cargar el dataset de incendios a normalizar

In [2]:
# Definir la ruta del dataset
dataset_path = r'C:\00 - Proyecto Incendios Galicia - END\data\02 - Municipio normalizado\02 - incendios\01 - historico incendios galicia municipio normalizado.csv'

# Cargar el archivo CSV
df = pd.read_csv(dataset_path)

print(f'Filas cargadas: {len(df)}')
print('Columnas disponibles:', list(df.columns))
print(f'Tamaño del dataset: {df.shape[0]} filas x {df.shape[1]} columnas')
df.head()

Filas cargadas: 114762
Columnas disponibles: ['campania', 'numeroparte', 'estado', 'comunidad', 'provincia', 'municipio', 'comarcaisla', 'entidadmenor', 'numeromunicipiosafectados', 'hoja', 'cuadricula', 'huso', 'coordenadax', 'coordenaday', 'datum', 'numeropuntosinicioincendio', 'detectado', 'extinguido', 'causa', 'motivacion', 'superficiearbolada', 'superficienoarbolada', 'superficietotalforestal', 'superficieagricola', 'otrassuperficiesnoforestales', 'afectozonasinterfazurbanoforestal', 'tipointerfazafectado', 'afectoespacioprotegido', 'afectotierrasagrarias', 'afectozar', 'numeropartepss']
Tamaño del dataset: 114762 filas x 31 columnas


C:\Users\Jacinto\AppData\Local\Temp\ipykernel_3472\4128871795.py:5: DtypeWarning: Columns (11,12,13) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(dataset_path)


,campania,numeroparte,estado,comunidad,provincia,municipio,comarcaisla,entidadmenor,numeromunicipiosafectados,hoja,...,superficienoarbolada,superficietotalforestal,superficieagricola,otrassuperficiesnoforestales,afectozonasinterfazurbanoforestal,tipointerfazafectado,afectoespacioprotegido,afectotierrasagrarias,afectozar,numeropartepss
0,2000.0,2.000150e+09,Migración,GALICIA,A CORUÑA,fisterra,FISTERRA,DUIO (SAN VICENTE),1.0,101.0,...,"0,1400","0,1400","0,0000","0,0000",Sin determinar,,No,No,No,
1,2000.0,2.000150e+09,Migración,GALICIA,A CORUÑA,boiro,BARBANZA,CURES (SANTO ANDRÉ),1.0,101.0,...,"0,1000","0,1000","0,0000","0,0000",Sin determinar,,No,No,No,
2,2000.0,2.000150e+09,Migración,GALICIA,A CORUÑA,monfero,FERROL,ALTO DE XESTOSO (O) (SANTA MARÍA),1.0,201.0,...,"0,0400","0,0400","0,0000","0,0000",Sin determinar,,No,No,No,
3,2000.0,2.000150e+09,Migración,GALICIA,A CORUÑA,ortigueira,FERROL,NEVES (AS) (SANTA MARÍA),1.0,201.0,...,"0,2000","0,2000","0,0000","0,0000",Sin determinar,,No,No,No,
4,2000.0,2.000150e+09,Migración,GALICIA,A CORUÑA,zas,FISTERRA,CARREIRA (SANTIAGO),1.0,101.0,...,"0,1200","0,1200","0,0000","0,0000",Sin determinar,,No,No,No,


In [3]:
# Duplicar la columna 'detectado' como 'fecha' para tener ambas referencias temporales
if 'detectado' in df.columns:
    df['fecha'] = df['detectado']
    print("Columna 'fecha' creada a partir de 'detectado'.")
else:
    print("La columna 'detectado' no existe en el DataFrame.")

Columna 'fecha' creada a partir de 'detectado'.


In [4]:
# Normalizar las columnas de fecha: 'detectado', 'extinguido' y 'fecha' (quitar hora y dejar solo fecha en formato YYYY-MM-DD)
for col in ['detectado', 'extinguido', 'fecha']:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], format='%d/%m/%Y %H:%M:%S', errors='coerce').dt.date
        print(f"Columna '{col}' normalizada. Ejemplo de valores:")
        print(df[col].dropna().astype(str).unique()[:5])

Columna 'detectado' normalizada. Ejemplo de valores:
['2000-01-19' '2000-01-20' '2000-01-21' '2000-01-22' '2000-01-23']
Columna 'extinguido' normalizada. Ejemplo de valores:
['2000-01-19' '2000-01-20' '2000-01-21' '2000-01-22' '2000-01-23']
Columna 'fecha' normalizada. Ejemplo de valores:
['2000-01-19' '2000-01-20' '2000-01-21' '2000-01-22' '2000-01-23']


In [5]:
# Crear la columna 'incendio' con valor 'si' en todas las filas
df['incendio'] = 'si'
print("Columna 'incendio' creada. Valores únicos:", df['incendio'].unique())

Columna 'incendio' creada. Valores únicos: ['si']


## 3. Completar la tabla con registros de incendio = 'no' para cada municipio y fecha sin incendio detectado
A continuación se generarán todas las combinaciones de municipio y fecha (del 1 de enero de 2000 al 31 de diciembre de 2022) y se marcarán como 'incendio = no' aquellas combinaciones que no existen ya en el dataset.

In [6]:
# 1. Obtener todos los municipios únicos y el rango de fechas
municipios = df['municipio'].unique()
fecha_inicio = datetime.date(2000, 1, 1)
fecha_fin = datetime.date(2022, 12, 31)
fechas = pd.date_range(fecha_inicio, fecha_fin).date

# 2. Crear DataFrame con todas las combinaciones municipio-fecha
combinaciones = pd.MultiIndex.from_product([municipios, fechas], names=['municipio', 'detectado'])
df_completo = pd.DataFrame(index=combinaciones).reset_index()

# 3. Marcar las combinaciones que ya existen en el dataset original
df['detectado'] = pd.to_datetime(df['detectado']).dt.date
df['clave'] = list(zip(df['municipio'], df['detectado']))
claves_existentes = set(df['clave'])
df_completo['clave'] = list(zip(df_completo['municipio'], df_completo['detectado']))
df_completo['incendio'] = df_completo['clave'].apply(lambda x: 'si' if x in claves_existentes else 'no')

# 4. Eliminar la columna clave auxiliar
df_completo.drop(columns=['clave'], inplace=True)

# 5. Concatenar solo los registros nuevos (incendio = 'no') al dataset original
df_no = df_completo[df_completo['incendio'] == 'no'].copy()
df_final = pd.concat([df, df_no], ignore_index=True)

print(f"Registros originales: {len(df)}")
print(f"Registros añadidos (incendio = 'no'): {len(df_no)}")
print(f"Total de registros tras completar: {len(df_final)}")

Registros originales: 114762
Registros añadidos (incendio = 'no'): 2561293
Total de registros tras completar: 2676055


In [7]:
# Guardar el dataset final en la ruta indicada

ruta_export = r'C:\00 - Proyecto Incendios Galicia - END\data\03 - Normalizado completo\02 - incendios'
os.makedirs(ruta_export, exist_ok=True)
archivo_export = os.path.join(ruta_export, '01 - incendios normalizado completo.csv')
df_final.to_csv(archivo_export, index=False, encoding='utf-8')
print(f'Dataset final guardado en: {archivo_export}')

Dataset final guardado en: C:\00 - Proyecto Incendios Galicia - END\data\03 - Normalizado completo\02 - incendios\01 - incendios normalizado completo.csv


In [8]:
# Revisar el dataset final exportado para comprobar tipos y valores únicos

archivo_export = r'C:\00 - Proyecto Incendios Galicia - END\data\03 - Normalizado completo\02 - incendios\01 - incendios normalizado completo.csv'
df_check = pd.read_csv(archivo_export)
print('Tipos de datos por columna:')
print(df_check.dtypes)
print('\nValores únicos por columna (primeros 10):')
for col in df_check.columns:
    print(f'\nColumna: {col}')
    print(df_check[col].unique()[:10])
print(f'\nNúmero total de filas: {len(df_check)}')
df_check.head()

C:\Users\Jacinto\AppData\Local\Temp\ipykernel_3472\1756036538.py:4: DtypeWarning: Columns (2,3,4,6,7,10,11,12,13,14,15,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,33) have mixed types. Specify dtype option on import or set low_memory=False.
  df_check = pd.read_csv(archivo_export)


Tipos de datos por columna:
campania                             float64
numeroparte                          float64
estado                                object
comunidad                             object
provincia                             object
municipio                             object
comarcaisla                           object
entidadmenor                          object
numeromunicipiosafectados            float64
hoja                                 float64
cuadricula                            object
huso                                  object
coordenadax                           object
coordenaday                           object
datum                                 object
numeropuntosinicioincendio            object
detectado                             object
extinguido                            object
causa                                 object
motivacion                            object
superficiearbolada                    object
superficienoarbolada       

,campania,numeroparte,estado,comunidad,provincia,municipio,comarcaisla,entidadmenor,numeromunicipiosafectados,hoja,...,otrassuperficiesnoforestales,afectozonasinterfazurbanoforestal,tipointerfazafectado,afectoespacioprotegido,afectotierrasagrarias,afectozar,numeropartepss,fecha,incendio,clave
0,2000.0,2.000150e+09,Migración,GALICIA,A CORUÑA,fisterra,FISTERRA,DUIO (SAN VICENTE),1.0,101.0,...,"0,0000",Sin determinar,,No,No,No,,2000-01-19,si,"('fisterra', datetime.date(2000, 1, 19))"
1,2000.0,2.000150e+09,Migración,GALICIA,A CORUÑA,boiro,BARBANZA,CURES (SANTO ANDRÉ),1.0,101.0,...,"0,0000",Sin determinar,,No,No,No,,2000-01-19,si,"('boiro', datetime.date(2000, 1, 19))"
2,2000.0,2.000150e+09,Migración,GALICIA,A CORUÑA,monfero,FERROL,ALTO DE XESTOSO (O) (SANTA MARÍA),1.0,201.0,...,"0,0000",Sin determinar,,No,No,No,,2000-01-19,si,"('monfero', datetime.date(2000, 1, 19))"
3,2000.0,2.000150e+09,Migración,GALICIA,A CORUÑA,ortigueira,FERROL,NEVES (AS) (SANTA MARÍA),1.0,201.0,...,"0,0000",Sin determinar,,No,No,No,,2000-01-19,si,"('ortigueira', datetime.date(2000, 1, 19))"
4,2000.0,2.000150e+09,Migración,GALICIA,A CORUÑA,zas,FISTERRA,CARREIRA (SANTIAGO),1.0,101.0,...,"0,0000",Sin determinar,,No,No,No,,2000-01-19,si,"('zas', datetime.date(2000, 1, 19))"


### Normalización de tipos tras la carga del CSV exportado
Ejecuta la siguiente celda para forzar los tipos de datos más habituales y evitar warnings.

In [9]:
# Cargar el CSV exportado forzando tipos y normalizando columnas clave


archivo_export = r'C:\00 - Proyecto Incendios Galicia - END\data\03 - Normalizado completo\02 - incendios\01 - incendios normalizado completo.csv'

# Cargar con low_memory=False para evitar warnings
df_check = pd.read_csv(archivo_export, low_memory=False)

# Convertir fechas
for col in ['detectado', 'extinguido']:
    if col in df_check.columns:
        df_check[col] = pd.to_datetime(df_check[col], errors='coerce').dt.date

# Convertir a numérico las columnas que deberían serlo (ajusta la lista según tus necesidades)
cols_numericas = [
    'campania', 'numeroparte', 'numeromunicipiosafectados', 'hoja',
    'superficiearbolada', 'superficienoarbolada', 'superficietotalforestal',
    'superficieagricola', 'otrassuperficiesnoforestales'
 ]
for col in cols_numericas:
    if col in df_check.columns:
        df_check[col] = pd.to_numeric(df_check[col].astype(str).str.replace(',', '.', regex=False), errors='coerce')

# Eliminar columna 'clave' si no es necesaria
if 'clave' in df_check.columns:
    df_check = df_check.drop(columns=['clave'])

# Revisar tipos finales
print('Tipos de datos tras normalización:')
print(df_check.dtypes)
df_check.head()

Tipos de datos tras normalización:
campania                             float64
numeroparte                          float64
estado                                object
comunidad                             object
provincia                             object
municipio                             object
comarcaisla                           object
entidadmenor                          object
numeromunicipiosafectados            float64
hoja                                 float64
cuadricula                            object
huso                                  object
coordenadax                           object
coordenaday                           object
datum                                 object
numeropuntosinicioincendio            object
detectado                             object
extinguido                            object
causa                                 object
motivacion                            object
superficiearbolada                   float64
superficienoarbolada

,campania,numeroparte,estado,comunidad,provincia,municipio,comarcaisla,entidadmenor,numeromunicipiosafectados,hoja,...,superficieagricola,otrassuperficiesnoforestales,afectozonasinterfazurbanoforestal,tipointerfazafectado,afectoespacioprotegido,afectotierrasagrarias,afectozar,numeropartepss,fecha,incendio
0,2000.0,2.000150e+09,Migración,GALICIA,A CORUÑA,fisterra,FISTERRA,DUIO (SAN VICENTE),1.0,101.0,...,0.0,0.0,Sin determinar,,No,No,No,,2000-01-19,si
1,2000.0,2.000150e+09,Migración,GALICIA,A CORUÑA,boiro,BARBANZA,CURES (SANTO ANDRÉ),1.0,101.0,...,0.0,0.0,Sin determinar,,No,No,No,,2000-01-19,si
2,2000.0,2.000150e+09,Migración,GALICIA,A CORUÑA,monfero,FERROL,ALTO DE XESTOSO (O) (SANTA MARÍA),1.0,201.0,...,0.0,0.0,Sin determinar,,No,No,No,,2000-01-19,si
3,2000.0,2.000150e+09,Migración,GALICIA,A CORUÑA,ortigueira,FERROL,NEVES (AS) (SANTA MARÍA),1.0,201.0,...,0.0,0.0,Sin determinar,,No,No,No,,2000-01-19,si
4,2000.0,2.000150e+09,Migración,GALICIA,A CORUÑA,zas,FISTERRA,CARREIRA (SANTIAGO),1.0,101.0,...,0.0,0.0,Sin determinar,,No,No,No,,2000-01-19,si


In [11]:
# Guardar el DataFrame normalizado con tipos corregidos en un nuevo archivo
ruta_export_final = r'C:\00 - Proyecto Incendios Galicia - END\data\03 - Normalizado completo\02 - incendios'
archivo_export_final = ruta_export_final + '\\02 - incendios normalizado completo final.csv'
df_check.to_csv(archivo_export_final, index=False, encoding='utf-8')
print(f'Dataset final normalizado guardado en: {archivo_export_final}')

Dataset final normalizado guardado en: C:\00 - Proyecto Incendios Galicia - END\data\03 - Normalizado completo\02 - incendios\02 - incendios normalizado completo final.csv


## 4. AMPLIACIÓN DEL DATASET: Variables temporales y de festivos

En esta sección ampliaremos el dataset con las siguientes columnas adicionales:
- **estacionalidad**: Estación del año (primavera, verano, otoño, invierno)
- **fin_de_semana**: Si es fin de semana (sí/no)
- **festivo_autonomico**: Si es festivo autonómico de Galicia (sí/no)
- **festivo_nacional**: Si es festivo nacional de España (sí/no)

In [12]:
# Cargar el dataset actual de incendios para ampliarlo
import pandas as pd
import numpy as np
from datetime import datetime, date

# Verificar e instalar holidays si es necesario
try:
    import holidays
    print("Librería 'holidays' disponible")
except ImportError:
    print("Instalando librería 'holidays'...")
    import subprocess
    import sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'holidays'])
    import holidays
    print("Librería 'holidays' instalada exitosamente")

# Ruta del dataset actual
dataset_path = r'C:\00 - Proyecto Incendios Galicia - END\data\03 - Normalizado completo\02 - incendios\02 - incendios normalizado completo final.csv'

print("CARGANDO DATASET ACTUAL DE INCENDIOS")
print("=" * 60)

try:
    # Cargar el dataset
    df_incendios = pd.read_csv(dataset_path, low_memory=False)
    
    print(f"Dataset cargado exitosamente")
    print(f"Dimensiones: {df_incendios.shape[0]:,} filas x {df_incendios.shape[1]} columnas")
    print(f"Columnas actuales: {list(df_incendios.columns)}")
    
    # Verificar que existe la columna de fecha
    columnas_fecha = [col for col in df_incendios.columns if 'fecha' in col.lower() or 'detectado' in col.lower()]
    if columnas_fecha:
        col_fecha = columnas_fecha[0]
        print(f"Columna de fecha identificada: '{col_fecha}'")
    else:
        print("No se encontró columna de fecha")
        col_fecha = None
        
except Exception as e:
    print(f"Error cargando el dataset: {e}")
    df_incendios = None

Librería 'holidays' disponible
CARGANDO DATASET ACTUAL DE INCENDIOS
Dataset cargado exitosamente
Dimensiones: 2,676,055 filas x 33 columnas
Columnas actuales: ['campania', 'numeroparte', 'estado', 'comunidad', 'provincia', 'municipio', 'comarcaisla', 'entidadmenor', 'numeromunicipiosafectados', 'hoja', 'cuadricula', 'huso', 'coordenadax', 'coordenaday', 'datum', 'numeropuntosinicioincendio', 'detectado', 'extinguido', 'causa', 'motivacion', 'superficiearbolada', 'superficienoarbolada', 'superficietotalforestal', 'superficieagricola', 'otrassuperficiesnoforestales', 'afectozonasinterfazurbanoforestal', 'tipointerfazafectado', 'afectoespacioprotegido', 'afectotierrasagrarias', 'afectozar', 'numeropartepss', 'fecha', 'incendio']
Columna de fecha identificada: 'detectado'


In [13]:
if df_incendios is not None and col_fecha is not None:
    
    print("\nPREPARANDO FUNCIONES PARA NUEVAS VARIABLES")
    print("=" * 60)
    
    # Convertir la columna de fecha a datetime
    df_incendios[col_fecha] = pd.to_datetime(df_incendios[col_fecha], errors='coerce')
    
    # 1. FUNCIÓN PARA CALCULAR ESTACIONALIDAD
    def calcular_estacionalidad(fecha):
        """Calcula la estación del año basada en la fecha"""
        if pd.isna(fecha):
            return None
        
        mes = fecha.month
        dia = fecha.day
        
        # Definir estaciones del año (hemisferio norte)
        if (mes == 3 and dia >= 21) or mes in [4, 5] or (mes == 6 and dia <= 20):
            return 'primavera'
        elif (mes == 6 and dia >= 21) or mes in [7, 8] or (mes == 9 and dia <= 22):
            return 'verano'
        elif (mes == 9 and dia >= 23) or mes in [10, 11] or (mes == 12 and dia <= 20):
            return 'otoño'
        else:
            return 'invierno'
    
    # 2. FUNCIÓN PARA CALCULAR FIN DE SEMANA
    def es_fin_de_semana(fecha):
        """Determina si una fecha es fin de semana"""
        if pd.isna(fecha):
            return None
        # weekday(): lunes=0, domingo=6
        return 'si' if fecha.weekday() >= 5 else 'no'
    
    print("Funciones de cálculo temporal definidas")
    
else:
    print("No se puede continuar sin dataset o columna de fecha")


PREPARANDO FUNCIONES PARA NUEVAS VARIABLES
Funciones de cálculo temporal definidas


In [14]:
if df_incendios is not None and col_fecha is not None:
    
    print("\nDEFINIENDO FESTIVOS ESPAÑOLES Y GALLEGOS")
    print("=" * 60)
    
    # Instalar holidays si no está disponible
    try:
        import holidays
    except ImportError:
        print("Instalando librería 'holidays'...")
        import subprocess
        subprocess.check_call(['pip', 'install', 'holidays'])
        import holidays
    
    # 3. FESTIVOS NACIONALES DE ESPAÑA
    spain_holidays = holidays.Spain(years=range(2000, 2023))
    
    def es_festivo_nacional(fecha):
        """Determina si una fecha es festivo nacional español"""
        if pd.isna(fecha):
            return None
        return 'si' if fecha.date() in spain_holidays else 'no'
    
    # 4. FESTIVOS AUTONÓMICOS DE GALICIA
    # Estos son los festivos específicos de Galicia
    festivos_galicia = {
        # Día de las Letras Gallegas (17 de mayo)
        'letras_gallegas': (5, 17),
        # Día de Galicia (25 de julio) - Santiago Apóstol
        'dia_galicia': (7, 25),
    }
    
    def es_festivo_autonomico(fecha):
        """Determina si una fecha es festivo autonómico de Galicia"""
        if pd.isna(fecha):
            return None
        
        mes = fecha.month
        dia = fecha.day
        
        # Verificar festivos autonómicos
        for nombre, (mes_fest, dia_fest) in festivos_galicia.items():
            if mes == mes_fest and dia == dia_fest:
                return 'si'
        
        return 'no'
    
    print("Festivos nacionales: Configurados para España (2000-2022)")
    print("Festivos autonómicos: Día de las Letras Gallegas, Día de Galicia")
    
else:
    print("No se puede continuar sin dataset o columna de fecha")


DEFINIENDO FESTIVOS ESPAÑOLES Y GALLEGOS
Festivos nacionales: Configurados para España (2000-2022)
Festivos autonómicos: Día de las Letras Gallegas, Día de Galicia


In [15]:
if df_incendios is not None and col_fecha is not None:
    
    print("\nAPLICANDO FUNCIONES Y CREANDO NUEVAS COLUMNAS")
    print("=" * 60)
    
    # Hacer una copia del dataset para trabajar
    df_ampliado = df_incendios.copy()
    
    print("Calculando estacionalidad...")
    df_ampliado['estacionalidad'] = df_ampliado[col_fecha].apply(calcular_estacionalidad)
    
    print("Calculando fines de semana...")
    df_ampliado['fin_de_semana'] = df_ampliado[col_fecha].apply(es_fin_de_semana)
    
    print("Calculando festivos nacionales...")
    df_ampliado['festivo_nacional'] = df_ampliado[col_fecha].apply(es_festivo_nacional)
    
    print("Calculando festivos autonómicos...")
    df_ampliado['festivo_autonomico'] = df_ampliado[col_fecha].apply(es_festivo_autonomico)
    
    print("\nRESUMEN DE NUEVAS COLUMNAS CREADAS:")
    print("=" * 60)
    
    nuevas_columnas = ['estacionalidad', 'fin_de_semana', 'festivo_nacional', 'festivo_autonomico']
    
    for columna in nuevas_columnas:
        print(f"\n{columna.upper()}:")
        valores_unicos = df_ampliado[columna].value_counts().sort_index()
        for valor, count in valores_unicos.items():
            porcentaje = (count / len(df_ampliado)) * 100
            print(f"   {valor}: {count:,} registros ({porcentaje:.2f}%)")
    
    print(f"\nDATASET AMPLIADO COMPLETADO")
    print(f"Dimensiones originales: {df_incendios.shape[0]:,} x {df_incendios.shape[1]}")
    print(f"Dimensiones ampliadas: {df_ampliado.shape[0]:,} x {df_ampliado.shape[1]}")
    print(f"Columnas añadidas: {len(nuevas_columnas)}")
    
else:
    print("No se puede continuar sin dataset o columna de fecha")


APLICANDO FUNCIONES Y CREANDO NUEVAS COLUMNAS
Calculando estacionalidad...
Calculando fines de semana...
Calculando festivos nacionales...
Calculando festivos autonómicos...

RESUMEN DE NUEVAS COLUMNAS CREADAS:

ESTACIONALIDAD:
   invierno: 660,723 registros (24.69%)
   otoño: 647,598 registros (24.20%)
   primavera: 670,607 registros (25.06%)
   verano: 697,127 registros (26.05%)

FIN_DE_SEMANA:
   no: 1,909,374 registros (71.35%)
   si: 766,681 registros (28.65%)

FESTIVO_NACIONAL:
   no: 2,608,936 registros (97.49%)
   si: 67,119 registros (2.51%)

FESTIVO_AUTONOMICO:
   no: 2,661,426 registros (99.45%)
   si: 14,629 registros (0.55%)

DATASET AMPLIADO COMPLETADO
Dimensiones originales: 2,676,055 x 33
Dimensiones ampliadas: 2,676,055 x 37
Columnas añadidas: 4


In [16]:
if df_incendios is not None and col_fecha is not None:
    
    print("\nGUARDANDO DATASET AMPLIADO")
    print("=" * 60)
    
    # Definir ruta de exportación
    ruta_export = r'C:\00 - Proyecto Incendios Galicia - END\data\03 - Normalizado completo\02 - incendios'
    os.makedirs(ruta_export, exist_ok=True)
    
    # Nombre del archivo ampliado
    archivo_ampliado = os.path.join(ruta_export, '03 - incendios normalizado completo extendido.csv')
    
    try:
        # Guardar el dataset ampliado
        df_ampliado.to_csv(archivo_ampliado, index=False, encoding='utf-8')
        
        print(f"Dataset ampliado guardado exitosamente")
        print(f"Ubicación: {archivo_ampliado}")
        print(f"Registros guardados: {len(df_ampliado):,}")
        print(f"Columnas totales: {len(df_ampliado.columns)}")
        
        # Mostrar muestra del dataset ampliado
        print(f"\nMUESTRA DEL DATASET AMPLIADO:")
        print("=" * 80)
        
        # Seleccionar columnas relevantes para mostrar
        columnas_muestra = [col_fecha, 'municipio', 'incendio', 'estacionalidad', 
                           'fin_de_semana', 'festivo_nacional', 'festivo_autonomico']
        columnas_disponibles = [col for col in columnas_muestra if col in df_ampliado.columns]
        
        print(df_ampliado[columnas_disponibles].head(10).to_string(index=False))
        
        print(f"\nVERIFICACIÓN FINAL:")
        print("-" * 40)
        print(f"Estacionalidad: {df_ampliado['estacionalidad'].nunique()} valores únicos")
        print(f"Fin de semana: {df_ampliado['fin_de_semana'].nunique()} valores únicos")
        print(f"Festivo nacional: {df_ampliado['festivo_nacional'].nunique()} valores únicos")
        print(f"Festivo autonómico: {df_ampliado['festivo_autonomico'].nunique()} valores únicos")
        
    except Exception as e:
        print(f"Error guardando el dataset: {e}")
        
else:
    print("No se puede guardar sin dataset válido")


GUARDANDO DATASET AMPLIADO
Dataset ampliado guardado exitosamente
Ubicación: C:\00 - Proyecto Incendios Galicia - END\data\03 - Normalizado completo\02 - incendios\03 - incendios normalizado completo extendido.csv
Registros guardados: 2,676,055
Columnas totales: 37

MUESTRA DEL DATASET AMPLIADO:
 detectado    municipio incendio estacionalidad fin_de_semana festivo_nacional festivo_autonomico
2000-01-19     fisterra       si       invierno            no               no                 no
2000-01-19        boiro       si       invierno            no               no                 no
2000-01-19      monfero       si       invierno            no               no                 no
2000-01-19   ortigueira       si       invierno            no               no                 no
2000-01-19          zas       si       invierno            no               no                 no
2000-01-19 val do dubra       si       invierno            no               no                 no
2000-01-20   ort